# Notebook 2: Feature Extraction

This notebook extracts **quantitative features** from the segmentation masks produced in Notebook 1. Features are computed at two levels:

1. **Organelle level** — one row per organelle (area, shape, position within acinus, etc.)
2. **Cell level** — one row per cell, aggregating organelle statistics within that cell (density, mean area, type fractions, etc.)

By the end of this notebook you will have:
- `mitochondria_properties.csv`
- `lipid_droplet_properties.csv`
- `peroxisome_properties.csv`
- `average_properties_per_cell.csv`

These CSVs are the inputs for Notebook 3 (Visualization & Analysis).


## 1. Setup

Point `DATA_PATH` at the folder containing the segmentation masks from Notebook 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from liv_zones.organelle import organelle_features
from liv_zones.cell import cell_features

# ── Paths ──────────────────────────────────────────────────────────────────
# This should be the same SAVE_PATH used in Notebook 1.
DATA_PATH = "sample_data/output"

# ── Scale ──────────────────────────────────────────────────────────────────
# Must match the scale used in Notebook 1.
SCALE = 14.4024  # pixels per micron

print(f"Reading masks from: {DATA_PATH}")
print(f"Scale: {SCALE} px/µm")

## 2. Organelle-level features

`organelle_features()` iterates over every organelle in each mask and computes:

| Column | Description |
|:---|:---|
| `area` | Surface area (µm²) |
| `perimeter` | Perimeter (µm) |
| `axis_major_length` | Length of the major axis (µm) |
| `axis_minor_length` | Length of the minor axis (µm) |
| `aspect_ratio` | `major / minor` — shape index (1 = circle) |
| `solidity` | Area / convex hull area — compactness measure |
| `cell_id` | ID of the cell this organelle belongs to |
| `boundry_dist` | Distance from this organelle to the cell edge (px) |
| `ascini_position` | Normalised position along the portal-central axis (−1 to +1) |
| `aspect_type_1/2/3` | Boolean — which morphological category this organelle falls in |

### The `scale` argument

Raw measurements from regionprops are in **pixels**. Dividing by `scale` converts lengths to **microns** and by `scale²` converts areas to **µm²**. Make sure you pass the same scale value you determined in Notebook 1.

### The `organelle_list` argument

Pass a list of strings containing 'mito', 'lipid', or 'perox' to control which organelles are processed. Comment out any that you don't need.

In [ ]:
organelle_features(
    path=DATA_PATH,
    scale=SCALE,
    organelle_list=["mito", "lipid", "perox"],
)

print("Organelle feature extraction complete!")

### Inspect the organelle CSVs

In [ ]:
mito_props = pd.read_csv(f"{DATA_PATH}/mitochondria_properties.csv", index_col=0)
print(f"Mitochondria count: {len(mito_props)}")
mito_props.head()

In [ ]:
ld_props = pd.read_csv(f"{DATA_PATH}/lipid_droplet_properties.csv", index_col=0)
print(f"Lipid droplet count: {len(ld_props)}")
ld_props.head()

In [ ]:
perox_props = pd.read_csv(f"{DATA_PATH}/peroxisome_properties.csv", index_col=0)
print(f"Peroxisome count: {len(perox_props)}")
perox_props.head()

## 3. Understanding organelle classification

Each organelle is assigned to one of **three morphological categories** based on a key shape parameter:

| Organelle | Property | Type 1 | Type 2 | Type 3 |
|:---|:---|:---|:---|:---|
| Mitochondria | Aspect ratio | Compact (< 1.35) | Intermediate | Elongated (≥ 1.97) |
| Lipid droplets | Area (µm²) | Small (< 0.88) | Medium | Large (≥ 2.28) |
| Peroxisomes | Aspect ratio | Round (< 1.33) | Intermediate | Elongated (≥ 1.84) |

The `aspect_type_1`, `aspect_type_2`, `aspect_type_3` (or `area_type_*` for lipid droplets) columns are Boolean flags indicating which category each organelle belongs to.

Let's visualise the type distributions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Mitochondria — aspect ratio
mito_counts = [
    mito_props["aspect_type_1"].sum(),
    mito_props["aspect_type_2"].sum(),
    mito_props["aspect_type_3"].sum(),
]
axes[0].bar(["Type 1\n(compact)", "Type 2\n(intermediate)", "Type 3\n(elongated)"],
            mito_counts, color=["#e74c3c", "#e67e22", "#2ecc71"])
axes[0].set_title("Mitochondria\nmorphological types", fontsize=11)
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Type (by aspect ratio)")

# Lipid droplets — area
ld_counts = [
    ld_props["area_type_1"].sum(),
    ld_props["area_type_2"].sum(),
    ld_props["area_type_3"].sum(),
]
axes[1].bar(["Type 1\n(small)", "Type 2\n(medium)", "Type 3\n(large)"],
            ld_counts, color=["#3498db", "#2980b9", "#1a5276"])
axes[1].set_title("Lipid droplets\nsize types", fontsize=11)
axes[1].set_ylabel("Count")
axes[1].set_xlabel("Type (by area)")

# Peroxisomes — aspect ratio
perox_counts = [
    perox_props["aspect_type_1"].sum(),
    perox_props["aspect_type_2"].sum(),
    perox_props["aspect_type_3"].sum(),
]
axes[2].bar(["Type 1\n(round)", "Type 2\n(intermediate)", "Type 3\n(elongated)"],
            perox_counts, color=["#f39c12", "#e67e22", "#d35400"])
axes[2].set_title("Peroxisomes\nmorphological types", fontsize=11)
axes[2].set_ylabel("Count")
axes[2].set_xlabel("Type (by aspect ratio)")

plt.suptitle("Organelle type distributions across the acinus", fontsize=13)
plt.tight_layout()
plt.show()

### Shape distributions

It's also useful to look at the raw distributions of the key shape parameters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(mito_props["aspect_ratio"].dropna(), bins=40, color="#e74c3c", edgecolor="white")
axes[0].axvline(1.352, color="black", linestyle="--", label="Type 1/2 split")
axes[0].axvline(1.970, color="gray",  linestyle="--", label="Type 2/3 split")
axes[0].set_title("Mitochondria aspect ratio", fontsize=11)
axes[0].set_xlabel("Aspect ratio (major / minor)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8)

axes[1].hist(ld_props["area"].dropna(), bins=40, color="#3498db", edgecolor="white")
axes[1].axvline(0.88,  color="black", linestyle="--", label="Type 1/2 split")
axes[1].axvline(2.283, color="gray",  linestyle="--", label="Type 2/3 split")
axes[1].set_title("Lipid droplet area (µm²)", fontsize=11)
axes[1].set_xlabel("Area (µm²)")
axes[1].set_ylabel("Count")
axes[1].legend(fontsize=8)

axes[2].hist(perox_props["aspect_ratio"].dropna(), bins=40, color="#f39c12", edgecolor="white")
axes[2].axvline(1.327, color="black", linestyle="--", label="Type 1/2 split")
axes[2].axvline(1.843, color="gray",  linestyle="--", label="Type 2/3 split")
axes[2].set_title("Peroxisome aspect ratio", fontsize=11)
axes[2].set_xlabel("Aspect ratio (major / minor)")
axes[2].set_ylabel("Count")
axes[2].legend(fontsize=8)

plt.suptitle("Shape parameter distributions with classification thresholds", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Cell-level aggregation

`cell_features()` reads the per-organelle CSVs produced above and aggregates them to produce **one row per cell**. For each organelle type and each morphological sub-type it computes:

| Property | Description |
|:---|:---|
| `*_density` | Number of organelles per µm² of cell area |
| `*_avg_area` | Mean area of organelles in this cell (µm²) |
| `*_percent_total_area` | Fraction of cell area occupied by this organelle type |
| `*_aspect_ratio` | Mean aspect ratio of organelles in this cell |
| `percent_type_*` | Fraction of organelles that belong to each morphological type |
| `*_distance_from_edge` | Mean relative radial position of organelles (0 = cell centre, 1 = edge) |
| `ascini_position` | Position of this cell along the portal-central axis (−1 to +1) |

In [ ]:
cell_features(
    path=DATA_PATH,
    scale=SCALE,
)

print("Cell-level feature extraction complete!")

In [ ]:
cell_props = pd.read_csv(f"{DATA_PATH}/average_properties_per_cell.csv", index_col=0)
print(f"Cells in dataset: {len(cell_props)}")
print(f"Columns:          {len(cell_props.columns)}")
cell_props.head()

## 5. Quick data sanity check

Before moving to visualisation, it's worth checking that the values are in a reasonable range.

In [ ]:
# Summary statistics for a selection of key columns
cols_of_interest = [
    "area", "ascini_position",
    "mito_density", "mito_avg_area", "mito_aspect_ratio",
    "ld_density", "ld_avg_area",
    "peroxisome_density", "peroxisome_avg_area",
]
# Only show columns that actually exist (depends on which organelles were processed)
cols_of_interest = [c for c in cols_of_interest if c in cell_props.columns]

cell_props[cols_of_interest].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Cell area distribution
axes[0].hist(cell_props["area"].dropna(), bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Cell area distribution", fontsize=11)
axes[0].set_xlabel("Area (µm²)")
axes[0].set_ylabel("Number of cells")

# Acinus position distribution
axes[1].hist(cell_props["ascini_position"].dropna(), bins=20,
             color="coral", edgecolor="white")
axes[1].set_title("Acinus position distribution", fontsize=11)
axes[1].set_xlabel("Acinus position (−1 = central vein, +1 = portal vein)")
axes[1].set_ylabel("Number of cells")
axes[1].axvline(0, color="black", linestyle="--", alpha=0.5, label="Midpoint")
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

The following CSV files are now in `sample_data/output/`:

| File | Contents |
|:---|:---|
| `mitochondria_properties.csv` | Per-organelle features for all mitochondria |
| `lipid_droplet_properties.csv` | Per-organelle features for all lipid droplets |
| `peroxisome_properties.csv` | Per-organelle features for all peroxisomes |
| `average_properties_per_cell.csv` | Aggregated statistics per cell |

**Next:** Open `03_visualization.ipynb` to explore how organelle properties vary along the portal-to-central axis.